# Setup

**Data**

[OSD-927 (Myeloid Cerebellum)](https://osdr.nasa.gov/bio/repo/data/studies/OSD-927), [OSD-928 (Myeloid Forebrain)](https://osdr.nasa.gov/bio/repo/data/studies/OSD-928), [OSD-929 (Myeloid Hippocampus)](https://osdr.nasa.gov/bio/repo/data/studies/OSD-929), [OSD-930 (Non-Myeloid Cerebellum)](https://osdr.nasa.gov/bio/repo/data/studies/OSD-930), [OSD-931 (Non-Myeloid Forebrain)](https://osdr.nasa.gov/bio/repo/data/studies/OSD-931), and [OSD-932 (Non-Myeloid Hippocampus)](https://osdr.nasa.gov/bio/repo/data/studies/OSD-932).

---

**Processing Raw Files**

See `create_preprocessed_927_928_929_930_931_932.ipynb`:

- Retrieves fastqz and metadata (e.g., `OSD-932_metadata_OSD-927-ISA.zip`) files from OSDR and stores them in whatever directory you specify in the `superdirec` variable. **Be sure to use this same `superdirec` specification in the code cell below this markdown cell.**
- Downloads this [reference genome](), processes it for `CellRanger`, and stores it in a subdirectory of whatever directory you specify in the `superdirec` variable.
- Creates `CellRanger` Bash syntax to preprocess files (copy-pasted into `scripts_cellranger/run_cellranger_929.sh`).

---

**Annotation Guide Files**

Retrieve "GL-DPPD-7111_Mmus_Brain_CellType_GeneMarkers.csv" from [this GitHub link](https://github.com/nasa/GeneLab_Data_Processing/blob/master/scRNAseq/10X_Chromium_3prime_Data/GeneLab_CellType_GeneMarkers/GL-DPPD-7111_GeneMarker_Files/GL-DPPD-7111_Mmus_Brain_CellType_GeneMarkers.csv) and save it in the same directory as this notebook. The GitHub path is `nasa/GeneLab_Data_Processing/scRNAseq/10X_Chromium_3prime_Data/GeneLab_CellType_GeneMarkers/GL-DPPD-7111_GeneMarker_Files`.

Use `GL-DPPD-7111_GeneMarker_Files/GL-DPPD-7111_Mmus_Brain_CellType_GeneMarkers_newest.csv`.

I added border-associated macrophage markers taken from [this paper](https://pmc.ncbi.nlm.nih.gov/articles/PMC12094687/) and [this paper](https://www.sciencedirect.com/science/article/pii/S1074761325001682).

I also added David's DAM microglia markers using this code to print the genes in the same format as the original marker file, and pasting the output in the marker file:
```
print("\n".join(pd.read_csv(
    "Glial_mouse_markers_microglia_keren-shaul-2017-davidp.csv").groupby(
        "cellName").apply(lambda x: x.name + "," + '"' + ",".join(
            x.geneSymbol) + '"')))

print("Oligodendrocyte (DAO),\"" + ",".join(pd.read_csv(
    "dol_genes.csv").stack().unique().tolist()) + "\"")
```

I further refined microglial and astrocyte markers using [this paper](https://www.cell.com/immunity/fulltext/S1074-7613(18)30485-0?_returnURL=https%3A%2F%2Flinkinghub.elsevier.com%2Fretrieve%2Fpii%2FS1074761318304850%3Fshowall%3Dtrue) and [this paper](https://doi.org/10.1038/s41598-023-39890-0).


Possible Map My Cells region keys = ["RHP", "RSP", "ACA", "PL-ILA-ORB", "AUD-TEa-PERI-ECT", "SS-GU-VISC", "MO-FRP", "PAL", "sAMY", "CTXsp", "HY", "STRv", "OLF", "LSX", "AI", "STRd", "VIS-PTLp", "VIS", "TH", "MOp", "ENT", "HIP", "P", "MB", "MY", "CB", "AUD", "SSp", "TEa-PERI-ECT"]

---

**Use Conda**

While in the folder containing this notebook:
`conda env create -f rapidsc.yml`



Navigate back to where you want to clone the `scflow` repository (I recommend home)
`cd`

Clone `scflow` from GitHub.
`git clone git@github.com:easlinger/scflow.git`

Navigate to the folder where `scflow` is:
`pip install .`

`pip install senepy`

---

**For NVIDIA Drivers (Linux)**

```
sudo apt update
sudo apt install -y build-essential dkms

sudo apt install -y wget
wget https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64/cuda-ubuntu2204.pin
sudo mv cuda-ubuntu2204.pin /etc/apt/preferences.d/cuda-repository-pin-600
sudo apt-key adv --fetch-keys https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64/3bf863cc.pub
sudo add-apt-repository "deb https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64/ /"

sudo apt update
sudo apt install -y cuda
```

---

To clear up CUDA memory:

```
import torch
import gc

gc.collect()
torch.cuda.empty_cache()
```

## Imports & Display

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import os
import re
import logging
import functools
import json
import cupy
import rmm
import warnings
from warnings import warn
try:
    import torch
    import gc
    torch.set_float32_matmul_precision("medium")
    gc.collect()
    torch.cuda.empty_cache()
except Exception:
    pass

# try:
#     import rapids_singlecell as rsc
# except Exception:
#     rsc = None

import ipynbname
import importlib
import matplotlib.pyplot as plt
import seaborn as sns
import anndata
import scanpy as sc
import scipy.sparse as sp
import pandas as pd
import numpy as np
import scflow

for pkg in ["torch", "anndata", "scanpy", "decoupler",
                "pandas", "numpy", "scipy", "scflow"]:
    print(f"{pkg}: {importlib.metadata.version(pkg)}")

pd.set_option("display.max_rows", 500)  # or None for unlimited rows
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)


class CategoricalFilter(logging.Filter):
    def filter(self, record):
        return "storing" not in record.getMessage() or \
               "as categorical" not in record.getMessage()


logger = logging.getLogger("anndata")
logger.addFilter(CategoricalFilter())

warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)


def write_h5ad_safely(adata, file_out, filter_genes=None, verbose=True):
    """Write `anndata` object to h5ad file."""
    if filter_genes is not None:
        print(f"Removing genes: {filter_genes}")
        adata = adata[:, ~adata.var_names.isin(filter_genes)].copy()
        if "gene_ids" in adata.var.columns:  # avoid h5 error
            adatas[x].var = adatas[x].var.drop("gene_ids", axis=1)
    if "tocsr" in dir(adata.X):
        if verbose is True:
            print("\t\t***Converting `.X` to CSR...")
        adata.X = adata.X.tocsr()
    for q in adata.layers:
        if sp.issparse(adata.layers[q]):
            if verbose is True:
                print(f"\t\t***Converting layer '{q}' to CSR...")
            adata.layers[q] = adata.layers[q].tocsr()
    if verbose is True:
        print(f"\n\n{'=' * 80}\nWriting to {file_out}\n{'=' * 80}\n\n")
    adata.write_h5ad(file_out)  # write

## Set Options (ACTIVELY SET THESE!)

**Note: The new h5ad data file will write to "data" sub-directory of where this notebook is.** 

(See line `direcs = [os.path.join(superdirec, i) for i in batches]`.)

--- 
**Examples of other ways of specifying options:**

`file_md` if from OSDR:

* `file_md = {i: f"{i}_metadata_{i}-ISA/s_{i}.txt" for i in batches}`

`file_timings`: Specify time of dissection for certain OSD datasets

* `file_timings = "RRRM-2 Dissection Timing_ISS-T[29][91].xlsx"`

`subcluster_biggest`

* `subcluster_biggest = 3  # sub-cluster biggest 3 clusters`
* `subcluster_biggest = 1  # sub-cluster biggest cluster`
* `subcluster_biggest = False  # no sub-clustering`

`kws_cluster`: May want to change based on variance ratio plots in "QC" section.

* `kws_cluster = dict(n_comps=50)  # cluster individual samples`
* `kws_cluster = None  # do not cluster individual samples`

`vars_regress_out`: To regress variables out of concatenated object

* `vars_regress_out = ["Time", "pct_counts_mt", "total_counts"]`

`covariates_categorical`

* `covariates_categorical = ["Comment[Euthanasia Date]"]`
* `covariates_continuous = ["Time"]`

For certain OSD datasets, age at beginning or region + cell type (material): 

* `col_age = "Factor Value[Age]"`
* `col_material = "Characteristics[Material Type]"`

`map_my_cells_region_keys`

* Possible Map My Cells region keys = ["RHP", "RSP", "ACA", "PL-ILA-ORB", "AUD-TEa-PERI-ECT", "SS-GU-VISC", "MO-FRP", "PAL", "sAMY", "CTXsp", "HY", "STRv", "OLF", "LSX", "AI", "STRd", "VIS-PTLp", "VIS", "TH", "MOp", "ENT", "HIP", "P", "MB", "MY", "CB", "AUD", "SSp", "TEa-PERI-ECT"]

`unannot_cts_share`: Cell types to remove from consideration during the annotation process. (Removes them from the marker dictionary.) Set to an empty list (`unannot_cts_share = []`) to use all cell types in the marker dictionary created from `file_mks_a_priori`.

`unannot_cts_batch`: Use to set batch-specific cell types to remove from consideration during the annotation process, for instance if you know certain cell types shouldn't be present in that batch because of cell sorting, tissue type, or anatomical region. Use an empty dictionary (`unannot_cts_batch = {}`) if you don't need any batch-specific cell types.

`prohib_cts`: Allow these cell types to be considered for annotation, but discard the clustering scheme if they are present.

`ondisk`: If `True`, use the on-disk concatenate function during integration to reduce memory usage. Results in writing individual sample files and a concatenated file, so be sure you have adequate disk space (and, if necessary to preserve disk space, move/remove those files once the overall integrated object is written to a file at the end of this script).

---

In [ ]:
# Batch Information
batches = ["OSD-927", "OSD-928", "OSD-929", "OSD-930", "OSD-931", "OSD-932"]
join_method = "inner"
remove_unexpressed_genes = False
min_cells_overall_sample = 3  # filter genes in overall sample

# Annotation Required, Prohibited, & Unconsidered Cell Types
non_myeloid = ["Neuron", "Excitatory", "Inhibitory",
               "Endothelial", "Astrocyte",
               "Oligodendrocyte", "Oligodendrocyte (DAO)",
               "OPC", "Neuroepithelial", "Pericyte"]
myeloid = ["Microglial", "Microglial (DAM)",
           "Macrophage (M1)", "Macrophage (BAM)",
           "Microglial (Pro-Inflammatory-M1)",
           "Microglial (Anti-Inflammatory-M2)"]
# non_myeloid_req = ["Neuron", "Endothelial", "Astrocyte",
#                    "Oligodendrocyte", "OPC"]
non_myeloid_req = ["Neuron", "Endothelial", "Astrocyte", "Oligodendrocyte"]
req_cts = {
    "OSD-927": ["Microglial"],
    "OSD-928": ["Microglial"],
    "OSD-929": ["Microglial"],
    "OSD-930": non_myeloid_req,
    "OSD-931": non_myeloid_req,
    "OSD-932": non_myeloid_req
}
unannot_cts_share = ["Macrophage (M1)",
                     "Macrophage (BAM)",
                     "Neuroepithelial"]
unannot_cts_batch = {
    "OSD-927": non_myeloid, "OSD-928": non_myeloid, "OSD-929": non_myeloid,
    "OSD-930": myeloid, "OSD-931": myeloid, "OSD-932": myeloid
}  # = {} if no batch-specific cell types to not consider for annotation
prohib_cts = []
collapse_neuron_subtypes = True
# remove_malat1 = True  # remove Malat1 before clustering
n_top_genes = 2000  # number of top genes to count as HVGs

# Set Data (& Meta-Data `file_md`) Sources & Species
superdirec = "/home/easlinger/data"  # directory with original data
species = "Mouse"
file_timings = None
file_md = {i: f"{superdirec}/{i}_metadata_{i}-ISA/s_{i}.txt" for i in batches}
file_md_assay = {i: str(f"{superdirec}/{i}_metadata_{i}-ISA/a_{i}_"
                        "transcription-profiling_single-cell-rna-sequencing_"
                        "Illumina.txt") for i in batches}

# Process
n_processors = os.cpu_count() - 1  # how many processors to use
overwrite = True  # allow overwrite of files?
ondisk = True  # use the on-disk concatenate function to reduce memory usage

# If You Want Results Emailed
# email = None
email = "elizabeth.aslinger@aya.yale.edu" # set email to None to skip
dir_results = f"outputs/{'_'.join(batches)}"
os.makedirs(dir_results, exist_ok=True)  # make output directory if needed
email = "elizabeth.aslinger@aya.yale.edu"  # set email to None to skip

# Set Source Data Directory & Output Options
file_concat = os.path.join("data", f"{'_'.join(batches)}_concatenated.h5ad")
file_new = os.path.join("data", f"{'_'.join(batches)}_integrated.h5ad")

# Set Sample & Batch IDs, Plus Other Potential Sources of Batch Effects
col_group = "Group"  # age &/or space flight
col_batch = "batch" if len(batches) > 1 else col_group
col_age = "Factor Value[Age]"
# col_age = "Factor Value[Age]"
col_condition = "Factor Value[Spaceflight]"
key_treatment = "Space Flight"
col_sample = "Source Name"  # sample column to use eventually
col_sample_original = "Sample Name"  # original sample column name in metadata
col_material = "Characteristics[Material Type]"  # split cell kind & region
col_region = "Brain_Region"  # will create from col_material
col_subject = "Subject"  # in case of replicates of subjects across batches

# ^ ...will split words in this column => columns "CellSort" & "Brain_Region"
# covariates_categorical = None
covariates_categorical = [col_age, col_region]  # or None
covariates_continuous = None

# Preprocessing/Clustering
subcluster_biggest = False  # no sub-clustering
kws_cluster = dict(
    n_comps=15, layer="log1p", use_highly_variable=True,
    kws_pca=dict(svd_solver="covariance_eigh")
)  # cluster individual samples
# kws_cluster = None  # don't cluster individual samples
vars_regress_out = None
remove_mt_genes = False

# Make Pre-Defined Marker Dictionary
file_mks_a_priori = "GL-DPPD-7111_Mmus_Brain_CellType_GeneMarkers_newest.csv"
# cts_superhierarchical = {
#     "Neuron": ["Excitatory", "Inhibitory", "Glutamatergic", "GABAergic",
#                "Dopaminergic", "Serotonergic", "Cholinergic"],
# }  # e.g., if classified as Neuron + other, just keep more specific type(s)
cts_superhierarchical = None
rename_marker_based_annotation = {
    "Excitatory | Inhibitory": "Excitatory-Inhibitory",
    "Inhibitory | Inhibitory": "Excitatory-Inhibitory",
    "Oligodendrocyte | Microglial | OPC": "Oligodendrocyte",
    str("Microglial | Microglial (DAM) | Microglial (Pro-Inflammatory-M1)"
        " | Microglial (Anti-Inflammatory-M2)"): "Microglial"
}  # e.g., allow for renaming as Exitatory-Inhibitory if no consensus on which

# Set Annotation Sources
model_celltypist = "Mouse_Whole_Brain.pkl"
map_my_cells_source = "WMB-10X" if species == "Mouse" else "WHB-10X" if (
    species == "Human") else None  # Map My Cells atlas source
# map_my_cells_region_keys = None
map_my_cells_region_keys = [
    "RSP", "ACA", "PL-ILA-ORB", "AUD-TEa-PERI-ECT", "SS-GU-VISC", "MO-FRP",
    "AI", "VIS-PTLp", "VIS", "MOp", "AUD", "SSp",
    "TEa-PERI-ECT"]  # regional subset for Map My Cells
map_my_cells_cell_keys = ["Isocortex"]  # pattern match: feature name column
source_patterns = ["Brain", "Cortical", "cortex"]  # for ToppGene

# # Cap VRAM
# rmm.reinitialize(managed_memory=True, pool_allocator=True,
#                  initial_pool_size=2 * 1024**3,
#                  maximum_pool_size=14 * 1024**3)

## Input Checks & Derived Settings

You might want to look over this code to make sure derived variables are what you want them to be, but it should work out of the box in most circumstances.

If you specified `file_timings`, you probably want to check if the format works with this code.

In [ ]:
#  Clustering Column Names
col_celltype = "leiden"  # key added/column name
col_l_i = "leiden_individual"
col_i = "annotation_by_markers_individual"
unlabeled_cat = "Heterogeneous"  # if can't find one best-fit cell label
cci_scanvi = col_i + "_heterogeneous_collapsed"
sep = " | "  # separator for heterogeneous annotations

# Current File & Data Directories
direcs = [os.path.join(superdirec, i) for i in batches]
cur_file = os.path.join(os.path.abspath(""), f"{ipynbname.name()}.ipynb")
html_out = os.path.join(dir_results, os.path.basename(
    os.path.splitext(cur_file)[0])) + ".html"

# A Priori Marker Dictionary
unannot_cts = {k: unannot_cts_share + list(unannot_cts_batch[k] if (
    k in unannot_cts_batch) else []) for k in batches}  # + shared => batches
mks_a_priori = pd.read_csv(file_mks_a_priori, index_col=0).iloc[:, 0]
mks_a_priori = dict(mks_a_priori.groupby(mks_a_priori.index.names[0]).apply(
    set)) if "," not in str(mks_a_priori.iloc[0]) else dict(
        mks_a_priori.apply(lambda x: set(x.split(","))))
# if cts_superhierarchical is not None:  # remove hierarchical cell types
#     _ = [mks_a_priori.pop(i, None) for i in cts_superhierarchical]

# Marker Dictionaries with Neuron Sub-Categories Collapsed
mks_collapsed = {**mks_a_priori}
if collapse_neuron_subtypes is True:
    if "Neuron" in mks_collapsed and "Excitatory" in mks_collapsed and \
            "Inhibitory" in mks_collapsed:
        mks_collapsed["Neuron"] = mks_collapsed["Neuron"].union(mks_collapsed[
            "Excitatory"]).union(mks_collapsed["Inhibitory"])
    _ = mks_collapsed.pop("Excitatory", None)
    _ = mks_collapsed.pop("Inhibitory", None)
print("\n\n".join([f"{i}: {mks_collapsed[i]}" for i in mks_collapsed]))

# Check Inputs
if model_celltypist == "Mouse_Whole_Brain.pkl" and species != "Mouse":
    raise ValueError("Manually set `model_celltypist` for non-mouse!")
if species != "Mouse":
    if "Mmus" in file_mks_a_priori:
        raise ValueError(
            f"`file_mks_a_priori` ({file_mks_a_priori}) not for species. "
            f"Set new file appropriate for species {species}.")

# Get Metadata
if file_md is not None:
    metadata = [pd.read_csv(
        os.path.join(superdirec, file_md[i]),
        sep=None, engine="python").set_index(col_sample_original).join(
            pd.read_csv(os.path.join(superdirec, file_md_assay[i]),
                        sep=None, engine="python").set_index(
                            col_sample_original)[["Raw Data File"]])
                for i in batches]  # list of metadata for each batch
    for u in np.arange(len(metadata)):  # combined age & condition variable
        metadata[u] = metadata[u].join(metadata[u].apply(
            lambda x: x[col_condition] + str(
                " | " + str(x[col_age]) + " Weeks" if (
                    col_age in metadata[u].columns and len(
                        metadata[u][col_age].unique()) > 1) else ""),
            axis=1).to_frame(col_group))  # add space flight (x age if have)
        if col_material is not None:
            metadata[u] = metadata[u].join(metadata[u][col_material].apply(
                lambda x: pd.Series(x.split(" "), index=[
                    "CellSort", col_region])))  # region & myeloid vs. not
else:
    metadata = None

# Load Sample Data

```
sample_cr_align = pd.Series(files).rename_axis("Sample").apply(
    lambda x: os.path.basename(x.split("/outs")[0])).to_frame(
        "SRX").reset_index().set_index("SRX")
md_chk = metadata_combined["Raw Data File"].apply(
    lambda x: x.split(",")[0]).apply(
        lambda x: x.split("_R1")[0].split("Seq_")[1]).to_frame(
            "SRX").reset_index().set_index("SRX")
md_chk = md_chk.join(sample_cr_align).set_index(
    col_batch, append=True).reorder_levels([1, 0]).groupby(
        col_batch).apply(lambda x: x.sort_index()).reset_index(0, drop=True)
md_chk
```

In [ ]:
%%time

# Create a Subdirectory of Working Directory for Data Outputs
os.makedirs("data", exist_ok=True)

# Detect If Subject Replicates across Batches
samples = pd.concat({os.path.basename(d): pd.Series([metadata[u][
    col_sample].loc[metadata[u].index[metadata[u]["Raw Data File"].apply(
        lambda q: q.split("_")[2]) == x][0]] for x in [
            i for i in os.listdir(d) if os.path.isdir(os.path.join(
                d, i))]]) for u, d in enumerate(direcs)}).reorder_levels(
                    [1, 0]).sort_index()
dups = bool(samples.reset_index(drop=True).duplicated().any())

# Load Data
adatas, files = {}, {}
for u, d in enumerate(direcs):  # iterate over directories (batches)
    subdirs = [i for i in os.listdir(d) if os.path.isdir(os.path.join(d, i))]
    for x in subdirs:  # iterate sample subdirectories within batch directory
        i_x = metadata[u].index[metadata[u]["Raw Data File"].apply(
            lambda q: q.split("_")[2]) == x]  # match file to sample
        if len(i_x) > 1:
            raise ValueError(f"More than one index found for {x}: {i_x}")
        sample = metadata[u][col_sample].loc[i_x[0]]
        subject = str(sample) if dups is True else None  # subject != sample?
        if dups is True:
            sample = sample + "_" + batches[u]
        files[sample] = os.path.join(d, x, "outs/filtered_feature_bc_matrix")
        adatas[sample] = sc.read_10x_mtx(files[sample])  # read anndata

        # Metadata to `.obs`
        if metadata is not None:  # sample-specific metadata if available
            samp_metadata = metadata[u].loc[i_x[0]]  # sample metadata
            for v in samp_metadata.index.values:  # metadata => .obs columns
                adatas[sample].obs.loc[:, v] = samp_metadata.loc[v]
        if col_batch is not None:  # if a batch column specified
            adatas[sample].obs.loc[:, col_batch] = batches[u]  # batch => .obs
        if subject is not None:
            adatas[sample].obs.loc[:, col_subject] = subject
        adatas[sample].obs.loc[:, col_sample] = sample  # sample ID => .obs
        adatas[sample].obs.loc[:, f"n_cells_original_{col_sample}"] = adatas[
            sample].obs.shape[0]  # original number of cells
        print(adatas[sample])

print(files)
files_individual = dict(zip(files.keys(), [os.path.join(
    "data", f"{x}_processed.h5ad") for x in files]))  # new individual files
print(files_individual)
metadata_combined = None if metadata is None else pd.concat(
    metadata, keys=batches, names=[col_batch])
metadata_combined

# QC

## Perform Sample-Specific QC

Mouse pairing weird technical variation?

```
tmp2 = samples.rename_axis(["", col_batch]).to_frame(
    "subject").reset_index(0, drop=True).reset_index().set_index("subject")
tmp2 = tmp2.reset_index().assign(**{"Source Name": tmp2.reset_index().apply(
    lambda x: x["subject"] + "_" + x[col_batch], axis=1)})
tmp2.set_index("Source Name", inplace=True)
tmp = descriptives.stack().to_frame("V").join(tmp2).set_index([
    "batch", "subject"], append=True).reset_index(
        "Source Name", drop=True).loc[:, "pct_counts_mt", :]
tmp = tmp.V.unstack("Metric")[["50%"]]
tmp = tmp.join(tmp.groupby("subject").apply(
    lambda x: x.name.split("_")[0]).to_frame("Condition"))
tmp = tmp.join(tmp.groupby("subject").apply(lambda x: "_".join(x.name.split(
    "_")[1:])).to_frame("s")).set_index(["Condition", "s"], append=True)
tmp["50%"].reset_index([col_condition, "subject"], drop=True).reorder_levels(
    [0, 2, 1]).sort_index().unstack("s").unstack(col_batch)
```

In [ ]:
%matplotlib inline

plot_qc = False  # change to True to get sample-level QC plots (a bit slow)
qcs, n_cells_by_counts, descriptives, figs = scflow.pp.perform_qc_multi(
    adatas, col_batch=col_batch, col_sample=col_sample, plot=plot_qc,
    percentiles=[0.025, 0.10, 0.25, 0.50, 0.75, 0.85, 0.90, 0.975],
    figsize=(10, 10))  # perform QC on individual samples
for x in qcs:  # iterate QC metrics % plot percentiles by group
    fig = sns.catplot(qcs, y=x, hue=col_batch, kind="violin")
    fig.fig.suptitle(x)
    fig = sns.catplot(descriptives.loc[:, :, x][[
        i for i in descriptives if ("%" in i)]].stack().to_frame("Value"),
                      x="Metric", y="Value", kind="bar",
                      hue=col_batch, height=10)
    fig.fig.suptitle(x)
# qcs["total_counts"].describe()
# descriptives.stack().unstack("Variable")["pct_counts_mt"].unstack(-1)[
#     "75%"].sort_values()
# descriptives.stack().unstack("Variable")["total_counts"].unstack(-1)[
#     "75%"].sort_values()
# descriptives.stack().unstack("Variable")["n_genes_by_counts"].unstack(-1)[
#     "90%"].sort_values()
# descriptives.stack().unstack("Variable")["n_cells_by_counts"].unstack(-1)[
#     "75%"].sort_values()
descriptives.stack().unstack("Variable").round()

## Auto-Detect Filtering Thresholds

Use 2.5th and/& 97.5th percentile (sample-specific) as minimum genes per cell and minimum and maximum total counts (subject to specified absolute minima). Use 97.5th percentile as upper bound for percent mitochondrial count. 

Use an absolute minimum cells per gene.

Also include arguments to run a PCA on individual samples before integrating.

In [ ]:
# Options
bounds = descriptives[["2.5%", "97.5%"]].apply(lambda x: list(
    x), axis=1).unstack("Variable")  # list top/bottom 5% (~sample, variable)
abs_min_genes = 200  # regardless of %ile, minimum genes to retain cell
abs_min_count = 500  # regardless of %ile, minimum counts to retain cell
abs_max_mt = 30  # regardless of %ile, absolute maximum mitochondrial content
abs_min_cells = None  # don't want to filter genes in individual samples

# Set Thresholds
kws_pp = {}
for x in adatas:
    b_x = bounds.loc[x]
    b_counts =  b_x["total_counts"] if isinstance(b_x[
            "total_counts"], list) else b_x["total_counts"].iloc[0]
    b_counts = [max(b_counts[0], abs_min_count), b_counts[1]]
    kws_pp[x] = {
        "min_max_genes": [max((b_x["n_genes_by_counts"] if isinstance(
            b_x["n_genes_by_counts"], list) else b_x[
                "n_genes_by_counts"].iloc[0])[0], abs_min_genes), None],
        "min_max_cells": [abs_min_cells, None],
        # "min_max_cells": [max((b_x["n_cells_by_counts"] if isinstance(
        #     b_x["n_genes_by_counts"], list) else b_x[
        #         "n_genes_by_counts"].iloc[0])[0], abs_min_cells), None],
        "max_mt": min(abs_max_mt, (b_x["pct_counts_mt"] if isinstance(b_x[
            "pct_counts_mt"], list) else b_x["pct_counts_mt"].iloc[0])[1]),
        # "max_mt": abs_max_mt,
        "min_max_counts": b_counts,
        # "vars_regress_out": ["total_counts"],
        "target_sum": 1e4,
        "zero_center": True, "max_value": 10,  # scaling
        "n_top_genes": n_top_genes,
        "doublet_detection": "drop"
    }
print("\n".join([f"{s}: {kws_pp[s]}" for s in kws_pp]))
pd.DataFrame(kws_pp).T

# Preprocess & Cluster Individual

## Filter & Normalize

In [ ]:
# Preprocess
if overwrite is False:
    raise ValueError("Must be able to overwrite to use on-disk option")
var_names = []  # to store genes not filtered out for each sample
for x in files:  # iterate sample files
    print(f"\n\n{'=' * 80}\n{x}\n{'=' * 80}")
    adatas[x].obs.loc[:, f"kws_pp_{col_sample}"] = str(kws_pp[x])  # store kws
    adatas[x] = scflow.pp.preprocess(adatas[x], **kws_pp[x], plot_qc=False)
    var_names += [set(adatas[x].var_names)]  # track what genes still present

# Decide Join Method
# shared_gs = set.intersection(*var_names)  # genes in all after filtering
# all_gs = set.union(*var_names)  # genes in any post-filter sample
# print(f"\n\n{len(shared_gs)} genes present in all samples post-filtering "
#       f"(out of {len(all_gs)} total genes in any post-filter sample)\n\n")
# join_method = "outer"

# Number of Cells
n_cells = pd.DataFrame({x: pd.Series({
    "n_cells_original": adatas[x].obs[f"n_cells_original_{col_sample}"].iloc[
        0], "n_cells": adatas[x].obs.shape[0]}) for x in adatas}).T
n_cells = n_cells.assign(percent_cells_retained=100 * round(n_cells[
    "n_cells"] / n_cells["n_cells_original"], 2))
print(f"\n\n\n{'=' * 80}\n# Cells Post-Processing\n{'=' * 80}\n\n")
n_cells

## Examine/Test Preprocessing

Unit tests

In [ ]:
for p, ann in zip([kws_pp[x] for x in kws_pp], [adatas[x] for x in adatas]):
    print(f"\n\n{'=' * 80}\n{x}\n{'=' * 80}\n")
    assert all(ann.var["n_cells_by_counts"] >= p["min_max_cells"][0]) if (
        kws_pp[x]["min_max_cells"][0]) else True
    assert all(ann.var["n_cells_by_counts"] <= p[
        x]["min_max_cells"][1]) if kws_pp[x]["min_max_cells"][1] else True
    assert all(ann.obs["n_genes_by_counts"] >= p["min_max_genes"][0])
    assert all(ann.obs["n_genes_by_counts"] <= p[
        "min_max_genes"][1]) if p["min_max_genes"][1] else True
    assert all(ann.obs["pct_counts_mt"] <= p["max_mt"])
    assert all(ann.obs["total_counts"] >= p["min_max_counts"][0])
    assert all(ann.obs["total_counts"] <= p["min_max_counts"][1]) if (
        p["min_max_counts"][1]) else True
    print(p)
    print(ann.obs[["n_genes", "pct_counts_mt", "total_counts"]
                  ].describe().loc[["min", "max"]])
    print(ann.var[["n_cells_by_counts"]].describe().loc[[
        "min", "max"]])

# Mitochondrial Genes
print(f"\n\n\n{'=' * 80}\nNumber of HVG MT Genes\n{'=' * 80}\n\n")
print(pd.Series({x: float(adatas[x].var.loc[[i for i in adatas[
    x].var_names if ("mt-" in i.lower())]].highly_variable.sum())
                 for x in adatas}))

## Final QC/Filtering

## Cluster Individual

Iterate different clustering parameters to ensure extraction of common cell types

Make sample-specific alterations to one sample's annotation

(Ignore the following example code)

```
tmp = adatas[x].obs.iloc[:, 55:68].join(adatas[x].obs[[cai]]).groupby(
    cai).describe().stack().loc[:, ["25%", "50%", "75%"], :].rename_axis(
        [cai, "Percentile"]).rename_axis("Variable", axis=1).stack()
sns.catplot(tmp.to_frame("Value"), y="Value", x="Percentile",
            hue=cai, col="Variable", col_wrap=4, kind="bar", sharey=False)
```

You can override the annotation method for specific samples if needed:
```
annotate_override = {
    "FL_LAR_19_OSD-931": "jaccard",
    "HC_LAR_19_OSD-930": "jaccard"
}
```

Or use same annotation method for all samples: `annotate_override = {}`

In [ ]:
%%time

# Options
files_to_run = [] if kws_cluster is None else list(files.keys())
annotate = "overlap_coef"  # scanpy.tl.marker_gene_overlap method...
# ...set `annotate` to False to skip annotation
# annotate_override = {
#     "FL_LAR_20_OSD-927": "jaccard",
#     "FL_LAR_19_OSD-931": "jaccard"
# }  # ...can override method for specific samples
annotate_override = {}  # ...don't override method for specific samples
criteria = dict(adj_pval_threshold=0.01)  # ...criterion for annotation
# criteria = dict(top_n_markers=100)  # ...with these criteria for annotation
# ress = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 1, 1.2, 1.5]
ress = [0.2, 0.4, 0.6, 1, 1.5]
# resn_list = {k: ress if ("non-myeloid" in adatas[k].obs.iloc[0][
#     col_material]) else [i for i in ress if i <= 0.5] + [1.5]
#              for k in files}  # specific resolutions by file name
resn_list = list(ress)  # not specific resolutions by file name
# dist_list = [0.5, 1.5, 0.3, 0.8, 0.1, 1.0, 2]
dist_list = [0.5]
# perform_all = False  # move on once valid clustering scheme accomplished
perform_all = True  # perform all resolution/min_dist combinations

# Unexpressed Genes to Remove (If Needed)
if remove_unexpressed_genes is True:
    g_0 = pd.DataFrame({x: adatas[x].var.n_cells_by_counts for x in adatas}
                       ).T.all().replace(False, np.nan).dropna().index.values
    print(f"Will remove {len(g_0)} genes not expressed in any sample "
          f"before writing file: {list(g_0)}")
else:
    g_0 = None

# Clustering
valid_clustering = {x: [] for x in files}
for w, x in enumerate(files_to_run):  # iterate sample files
    cupy.get_default_memory_pool().free_all_blocks()
    cupy.get_default_pinned_memory_pool().free_all_blocks()
    valid_cts = False
    print(f"\n\n{'=' * 80}\n{x} ({w + 1} / {len(files_to_run)})\n{'=' * 80}")
    if remove_mt_genes is True:
        adatas[x] = adatas[x][:, ~adatas[x].var_names.str.startswith((
            "MT-", "Mt-", "mt-"))]  # remove mitochondrial genes?
    resn_list_specific = resn_list if isinstance(
        resn_list, list) else resn_list[x]
    anm = annotate_override[x] if x in annotate_override else annotate
    adatas[x].obs.loc[:, "annotate_method"] = anm
    adatas[x].obs.loc[:, "annotate_criteria"] = str(criteria)
    for t in dist_list:  # iterate min_dist
        if valid_cts is True and perform_all is False:
            break  # stop loop if valid clustering scheme already found
        for r in resn_list_specific:  # loop resolution
            end_of_loop = t == dist_list[-1] and r == resn_list_specific[-1]
            if valid_cts is True and perform_all is False:
                break  # stop loop if valid clustering scheme already found
            c_i, cai = [f"{q}_res{r}dist{t}" for q in [col_l_i, col_i]]
            kws_cl = {"resolution": r, "min_dist": t, **kws_cluster}
            adatas[x] = scflow.pp.cluster(
                adatas[x], plot=False, col_celltype=c_i, **kws_cl)
            sc.tl.rank_genes_groups(
                adatas[x], c_i, key_added=f"rank_genes_groups_{c_i}",
                n_genes=None, copy=False)  # find markers/DEGs
            if annotate is False:
                valid_cts = perform_all is False or end_of_loop is True
                adatas[x].obs.loc[:, col_l_i] = adatas[x].obs[c_i]
                adatas[x].obs.loc[:, "kws_cluster_individual"] = str(kws_cl)
                continue
            rcts, pcts, ucts = [z[adatas[x].obs[col_batch].iloc[0]] if (
                isinstance(z, dict)) else z for z in [
                    req_cts, prohib_cts, unannot_cts]]  # batch-specific?
            if ucts is not None:
                mksci = {z: mks_collapsed[z] for z in mks_collapsed if (
                    z not in ucts)}  # remove certain 1s from annotation guide
            _ = scflow.pp.annotate_by_marker_overlap(
                adatas[x], mksci, col_celltype=c_i, col_celltype_new=cai,
                sep=sep, celltypes_superhierarchical=cts_superhierarchical,
                **criteria, method=anm, inplace=True)  # annotate
            if rename_marker_based_annotation is not None:
                if any((y in list(adatas[x].obs[cai])
                            for y in rename_marker_based_annotation)):
                    rns = [y for y in rename_marker_based_annotation if (
                        y in list(adatas[x].obs[cai]))]
                    print(f"\t***Renaming: {rns}...")
                adatas[x].obs.loc[:, cai] = adatas[x].obs[cai].replace(
                        rename_marker_based_annotation)  # re-name
            print(list(adatas[x].obs[cai].unique()))
            # sns.catplot(adatas[x].obs, y="pct_counts_mt", x=c_i,
            #             hue=c_i, kind="violin")
            if adatas[x].obs[cai].isna().mean() > 0.25:  # to much NA?
                adatas[x].obs.loc[:, "kws_cluster_individual"] = np.nan
                continue
            adatas[x].obs.loc[:, cci_scanvi] = adatas[x].obs[
                cai].apply(lambda x: unlabeled_cat if (sep in x) else x)
            valid_cts = rcts is None or all([
                q in adatas[x].obs[cai].unique() for q in rcts])
            valid_cts = valid_cts and (pcts is None or all([
                q not in adatas[x].obs[cai].unique() for q in pcts]))
            valid_cts = False if any(adatas[x].obs[
                cci_scanvi] == unlabeled_cat) else valid_cts
            # If Found Valid Clustering Scheme, Plot, Write, & Store Results
            if valid_cts is True:
                valid_clustering[x] += [(c_i, cai, kws_cl)]
                print(kws_cl)
                if perform_all is False:
                    adatas[x].obs.loc[:, col_l_i] = adatas[x].obs[c_i]
                    adatas[x].obs.loc[:, col_i] = adatas[x].obs[cai]
                    adatas[x].obs["kws_cluster_individual"] = str(kws_cl)
                    try:
                        # sc.pl.pca_variance_ratio(adatas[x], log=True)
                        sc.pl.umap(adatas[x], wspace=0.3, color=[c_i, cai])
                    except Exception as err:
                        print(f"Plotting failed for {x}: {err}")
                    if ondisk is True:
                        write_h5ad_safely(adatas[x], files_individual[x],
                                          filter_genes=g_0)
                        adatas[x] = None

# Detect Samples with No Valid Clustering Scheme
if kws_cluster is not None:
    nctsi = [x for x in valid_clustering if len(valid_clustering[x]) == 0]
    if len(nctsi) > 0:
        print(f"No Valid Clustering: {nctsi}")

# Assign Main Clustering Scheme
if perform_all is True:
    clusters = pd.concat({x: pd.concat([pd.Series({
        **i[-1], "clusters": adatas[x].obs[i[1]].unique(), "# leiden": len(
            adatas[x].obs[i[0]].unique()), "c_i": i[0], "cai": i[
                1], "kws_cl": str(i[-1])}) for i in valid_clustering[
                    x]], axis=1).T.set_index(["resolution", "min_dist"])
                          for x in valid_clustering}, names=[col_subject])
    clusters = clusters.join(clusters["clusters"].apply(
        len).to_frame("# clusters"))
    print("\n\n\n", clusters, "\n\n\n")
    clusters_minmax = clusters.groupby(col_subject).apply(lambda y: y[y[
        "# clusters"] >= max(y["# clusters"])].loc[y.name]).groupby(
            col_subject).apply(lambda y: y[y["# leiden"] <= min(
                y["# leiden"])].loc[y.name]).reset_index(
                    "resolution").groupby(col_subject).apply(lambda y: y[y[
                        "resolution"] <= min(y["resolution"])].loc[
                            y.name].iloc[0])
    for x in adatas:  # store chosen clustering scheme in default columns
        print(f"\n\n{'=' * 80}\n{x}\n{'=' * 80}")
        try:
            c_i, cai = [clusters_minmax.loc[x][k] for k in ["c_i", "cai"]]
            adatas[x].obs.loc[:, col_l_i] = adatas[x].obs[c_i]
            adatas[x].obs.loc[:, col_i] = adatas[x].obs[cai]
            adatas[x].obs.loc[:, cci_scanvi] = adatas[x].obs[
                cai].apply(lambda x: unlabeled_cat if (sep in x) else x)
            adatas[x].obs["kws_cluster_individual"] = str(clusters_minmax.loc[
                x]["kws_cl"])
        except Exception as err:
            print(f"Could not store clustering scheme for {x}: {err}")
        try:  # plot all valid clustering schemes: UMAP
            for q in [[w[y] for w in valid_clustering[x]] for y in [0, 1]]:
                sc.pl.umap(adatas[x], wspace=0.2 if "leiden" in q[0] else 0.5,
                           color=q, ncols=scflow.pl.square_grid(len(q))[1])
        except Exception as err:
            print(f"Plotting all valid clusterings failed for {x}: {err}")
        try:  # plot chosen clustering scheme
            sc.pl.umap(adatas[x], wspace=0.3, color=[c_i, cai])
            scflow.pl.plot_matrix_marsilea(
                adatas[x], genes=mksci, col_celltype=c_i)
            scflow.pl.plot_matrix_marsilea(
                adatas[x], genes=mksci, col_celltype=cai)
        except Exception as err:
            print(f"Plotting failed for {x}: {err}")
else:
    clusters_minmax = None if kws_cluster is None else pd.DataFrame({
        k: valid_clustering[k][-1] for k in valid_clustering}).T

# Write If Haven't Yet & Using On-Disk Method
if (perform_all is True or kws_cluster is None) and (
        ondisk is True):  # write if needed & didn't in loop
    for x in adatas:
        write_h5ad_safely(adatas[x], files_individual[x], filter_genes=g_0)
        adatas[x] = None

# Remove Genes If Not Done During Write
if ondisk is False and g_0 is not None:
    for x in adatas:
        adatas[x] = adatas[x][:, ~adatas[x].var_names.isin(g_0)].copy()

# Print Parameters Used
# adatas = {x: sc.read(files_individual[x]) for x in files_individual}
clusters_minmax

# Concatenate & Integrate

If you are concerned about hard drive space, you may want to delete `file_concat` and `files_individual` after running this cell.

If it makes it through writing the concatenated file (`file_concat`, if `ondisk=True`) and you're running out of memory for further steps, try inserting the following code before the call to `scflow.Rna`:

```

# Memory-Saving Parameters
kws_integrate["drop_non_hvgs"] = True
kws_integrate["retain_original"] = False
kws_integrate["out_file_processed"] = str(
    f"{os.path.splitext(file_concat)[0]}_processed.h5ad")

```

Then run the `sflow.Rna` line in the cell below (just that line). If it fails part-way through, you can pick up where it left off if it wrote the "..._processed.h5ad" file:

```

# Integration Options
cct_available = kws_cluster is not None and (
    mks_a_priori is not None)  # individual annotations available?
kws_vi = {"early_stopping": True,
          "batch_size": 1024,  # raise/lower if more/less than 16 GB VRAM
          "max_epochs": 100,
          "accelerator": "gpu",
          "categorical_covariate_keys": covariates_categorical,
          "continuous_covariate_keys": covariates_continuous,
          "n_latent": 40, "n_hidden": 400}  # scVI/scANVI arguments

kws_integrate = {
    "col_celltype": cci_scanvi if cct_available else None,
    "flavor": "scanvi",
    # "flavor": "scvi",
    # "flavor": "scanorama",
    # "flavor": "harmony",
    "n_top_genes": n_top_genes,
    "kws_pp": None, "kws_cluster": None,
    "vars_regress_out": vars_regress_out,
    "max_value": 10, "zero_center": True, "target_sum": 1e4,
    "join": join_method, "merge": "same",
    # "drop_non_hvgs": True,  # just for the integration part
    "drop_non_hvgs": False,
    "use_rapids": True,
    "min_cells": min_cells_overall_sample,
    "fill_value": np.nan if join_method == "outer" else None,
    "retain_original": False,
    # "out_file": file_concat if ondisk is True else None,
    "verbose": True
}
if kws_integrate["flavor"] != "harmony":
    kws_integrate.update(kws_vi)
    kws_integrate["col_batch"] = None  # suppress using batch as covariate
else:
    kws_integrate["col_sample"] = col_sample
    kws_integrate["col_subject"] = col_subject
    kws_integrate["col_batch"] = col_batch if len(batches) > 1 else None

# If scVI/scANVI Integration & Individual Annotations Available
if kws_integrate["flavor"] in ["scvi", "scanvi"] and cct_available is True:
    kws_integrate.update({"unlabeled_category": unlabeled_cat})

flavor = "scanvi"
verbose = False
retain_original = False
drop_nonhvgs = False
col_celltype = cci_scanvi if cct_available else None
kwargs = {**kws_integrate}

pkg = sc
adata = sc.read_h5ad(str(
    f"{os.path.splitext(file_concat)[0]}_processed.h5ad"))

layer_log1p = "log1p"
layer_counts = "counts"
layer_scaled = "scaled"
KWS_SCVI = [
    "max_epochs", "accelerator", "devices", "train_size", "validation_size",
    "shuffle_set_split", "batch_size", "datasplitter_kwargs", "plan_kwargs",
    "datamodule", "benchmark", "default_root_dir", "enable_checkpointing",
    "checkpointing_monitor", "num_sanity_val_steps", "enable_model_summary",
    "early_stopping", "early_stopping_monitor", "early_stopping_min_delta",
    "early_stopping_patience", "early_stopping_warmup_epochs", "logger",
    "early_stopping_mode", "enable_progress_bar", "progress_bar_refresh_rate",
    "simple_progress_bar", "log_every_n_steps", "learning_rate_monitor"
]
KWS_SCVI_MODEL = ["registry", "n_hidden", "n_latent", "n_layers",
                  "dropout_rate", "dispersion", "gene_likelihood",
                  "use_observed_lib_size", "latent_distribution"]
KWS_SCANVI = ["n_samples_per_label", "check_val_every_n_epoch",
            "adversarial_classifier"]

pkg.pp.highly_variable_genes(
    adata, n_top_genes=n_top_genes, layer=layer_log1p,
    flavor="cell_ranger", batch_key=col_sample,
    subset=False)  # find (optionally subset to) highly variable

# Batch Keys/Covariates
col_covs = col_sample if col_batch is None else [col_subject if (
col_subject is not None) else col_sample, col_batch]
ccs = col_covs if isinstance(col_covs, str) else " & ".join(col_covs)
print(f"\n>>>Integrating with respect to {ccs} ({flavor.upper()})...")
print(f"\t***Using {layer_counts} layer for {flavor}...")
kws_setup = dict(layer=layer_counts, batch_key=col_sample)
kss = ["size_factor_key", "categorical_covariate_keys",
       "continuous_covariate_keys"]
for k in [i for i in kss if i in kwargs]:
    kws_setup[k] = kwargs.pop(k)  # extract setup arguments
ckws_pr = [str(kws_setup[i]) for i in [
    "categorical_covariate_keys", "continuous_covariate_keys"] if (
        i in kws_setup)]  # covariate keyword arguments?
if len(ckws_pr) > 0:
    print(f"\t***Using {', '.join(ckws_pr)} as covariates...")
if "categorical_covariate_keys" not in kws_setup:
    kws_setup["categorical_covariate_keys"] = None if (
        isinstance(col_covs, str)) else col_covs[1]
kws_train = {}
for k in pd.unique([i for i in KWS_SCVI if i in kwargs]):
    kws_train[k] = kwargs.pop(k)  # extract shared training arguments
if flavor.lower() == "scanvi":  # scANVI setup
    kws_train_scanvi = {**kws_train}  # start with shared arguments
    for k in [i for i in KWS_SCANVI if i in kwargs]:
        kws_train_scanvi[k] = kwargs.pop(k)
    unlabeled = kwargs.pop("unlabeled_category", "Unlabeled")
    if "use_minified" in kwargs:
        kws_setup["use_minified"] = kwargs.pop(x)

scvi.model.SCVI.setup_anndata(adata, **kws_setup)  # setup data
kws_train_scvi = {**kws_train}  # start with shared arguments
for k in [i for i in ["load_sparse_tensor", "early_stopping"] if (
        i in kwargs)]:
    kws_train_scvi[k] = kwargs.pop(k)
kws_model = {i: kwargs[i] for i in kwargs if i in KWS_SCVI_MODEL}
print(f"\t***Setting up scVI model: {kws_model}...")
model = scvi.model.SCVI(adata, **kws_model)  # scVI or scANVI model
print(f"\t***Setting up scVI model: {kwargs}...")
model = scvi.model.SCVI(adata, **kwargs)  # scVI or scANVI model
print(f"\t***Traning scVI: {kws_train_scvi}...")
model.train(**kws_train_scvi)  # train model
if flavor.lower() == "scanvi":
    print(f"\t***Setting up scANVI model: {kwargs}...")
    vimodel = scvi.model.SCANVI.from_scvi_model(
        model, adata=adata, labels_key=col_celltype,
        unlabeled_category=unlabeled, **kwargs)
    print(f"\t***Traning scANVI: {kws_train_scanvi}...")
    vimodel.train(**kws_train_scanvi)
    adata.obsm[new_pca_key] = vimodel.get_latent_representation(adata)
    adata.obs.loc[:, "annotation_scanvi"] = vimodel.predict(
        adata)
adata.obsm[new_pca_key] = model.get_latent_representation()

adata.obsm["X_pca_old"] = adata.obsm["X_pca"].copy()
adata.obsm["X_pca"] = adata.obsm[new_pca_key].copy()
adata.X = adata.layers[layer_counts].copy()  # set back to counts layer

adata.write_h5ad(file_new)
del adata

self = scflow.Rna(
    file_concat if file_concat else adatas, col_sample=col_sample,
    col_subject=col_subject, col_batch=col_batch, integrated=True
)
```

---

Example code looking at senescence-related gene expression:

```
mks = [pd.read_csv(os.path.join("gene_sets", i)).dropna(
    how="all", axis=1).assign(Source_File=i) for i in os.listdir("gene_sets")]
mks = [x.assign(Gene_Set=x["Source_File"].iloc[0].split("_2025")[0]) if (
    "pathway" in x) else x for x in mks]
mks = pd.concat(mks).drop("Unnamed: 0", axis=1).drop("row_id", axis=1)
mks = mks[mks.symbol.isin(self.rna.var_names)]
mks = mks[["Gene_Set", "symbol"]].set_index("Gene_Set").groupby(
    "Gene_Set").apply(lambda x: x["symbol"].to_list())
self.rna.var["n_cells_by_counts"].loc[mks.loc["Senmayo"]].sort_values()
```

In [ ]:
%%time

# Integration Options
cct_available = kws_cluster is not None and (
    mks_a_priori is not None)  # individual annotations available?
kws_vi = {"early_stopping": True,
          "batch_size": 1024,  # raise/lower if more/less than 16 GB VRAM
          "max_epochs": 100,
          "accelerator": "gpu",
          "categorical_covariate_keys": covariates_categorical,
          "continuous_covariate_keys": covariates_continuous,
          "n_latent": 40, "n_hidden": 400}  # scVI/scANVI arguments
kws_integrate = {
    "col_celltype": cci_scanvi if cct_available else None,
    "flavor": "scanvi",
    # "flavor": "scvi",
    # "flavor": "scanorama",
    # "flavor": "harmony",
    "n_top_genes": n_top_genes,
    "kws_pp": None, "kws_cluster": None,
    "vars_regress_out": vars_regress_out,
    "max_value": 10, "zero_center": True, "target_sum": 1e4,
    "join": join_method, "merge": "same",
    # "drop_non_hvgs": True,  # just for the integration part
    "drop_non_hvgs": False,
    "use_rapids": False,
    "min_cells": min_cells_overall_sample,
    "fill_value": np.nan if join_method == "outer" else None,
    "retain_original": False,
    "conserve_memory": True,
    # "out_file": file_concat if ondisk is True else None,
    "verbose": True
}
if kws_integrate["flavor"] != "harmony":
    kws_integrate.update(kws_vi)
    kws_integrate["col_batch"] = None  # suppress using batch as covariate
else:
    kws_integrate["col_sample"] = col_sample
    kws_integrate["col_subject"] = col_subject
    kws_integrate["col_batch"] = col_batch if len(batches) > 1 else None
if kws_integrate["flavor"] in ["scvi", "scanvi"] and cct_available is True:
    kws_integrate.update({"unlabeled_category": unlabeled_cat})

# Concatenate Samples into One Object
adatas = (anndata.experimental.concat_on_disk if ondisk else anndata.concat)(
    *list([files_individual, file_concat] if ondisk is True else [adatas]),
    axis="obs", join=join_method, label=col_sample, index_unique="_",
    pairwise=False, fill_value=None
)  # concatenate either on-disk or in-memory
if ondisk is False and file_concat is not None:  # write concatenated?
    adatas.write_h5ad(file_concat)
    adatas = None

# Integrate & Store Integration Parameters in Object
self = scflow.Rna(
    file_concat if file_concat else adatas, col_sample=col_sample,
    col_subject=col_subject, col_batch=col_batch, kws_integrate=kws_integrate
)

# Write Files for Processed/Integrated Objects?
if overwrite is True or not os.path.exists(file_new):
    self.rna.write_h5ad(file_new)
adatas = None

# Display
print(self.rna)
self.rna.obs

# Set Up Marker Dictionaries

Have to remove genes dropped in filtering from marker dictionaries

`mksn` doesn't include clusters in `unannot_cts`

In [ ]:
%matplotlib inline

# Example Code to Reload Data If Starting from Here
# Would also have to run "Setup" section (stopping before Load Sample Data)
# self = scflow.Rna(
#     file_new, col_sample=col_sample, col_subject=col_subject,
#     col_batch=col_batch, integrated=True, col_celltype="leiden")

mksn = {z: [g for g in mks_collapsed[z] if g in self.rna.var_names]
        for z in mks_collapsed if (z not in unannot_cts)} if isinstance(
            unannot_cts, list) else {z: [g for g in mks_collapsed[z] if (
                g in self.rna.var_names)] for z in mks_collapsed}
mksn = {k: v for k, v in mksn.items() if len(
    v) > 0 and k not in unannot_cts_share}

mks_c = dict(zip(mks_collapsed, [mks_collapsed[x].intersection(
        self.rna.var_names) for x in mks_collapsed]))
mks_c = {k: [g for g in v if [g for v in mks_c.values() for g in v].count(
    g) == 1] for k, v in mks_c.items()}

mks_cc = {k: [x for x in mks_a_priori[k] if x in self.rna.var_names]
          for k in mks_a_priori if k not in unannot_cts_share}

# Cluster

Perform PCA, UMAP embedding, and Leiden clustering on the integrated object

Marker gene-related code looks at top markers by log2fold-change and adjusted p-value cutoffs and sorts by adjusted p-values. Plots for predefined marker expression by cluster (if available) and cluster DEGs are created. 

The following is alternative code to plot marker expression. Use `kind = ["heat", "dot"]` to get dot plots too.

```
markers_dict = dict(markers_df.groupby(cct).apply(
    lambda x: list(x.reset_index().names)))  # dictionary version of df
_ = self.plot(genes=markers_dict, figsize=(15, 15),
              layer="log1p", standard_scale="var", kind="heat")
```


In [ ]:
%%time

# Clustering Options
resolution, min_dist = 0.15, 0.3
# n_neighbors = 50
n_neighbors = 50
t_p = 1e-5  # adjusted p-value threshold for DEGs/markers
t_lfc = 1.5  # logfold-change threshold for DEGs/markers
use_rapids = True

# Set Default Cell Type Column
cct = col_celltype  # because we want a shortened version for code readability
self._info["col_celltype"] = col_celltype

# Clustering
kws_cl_overall = dict(
    resolution=resolution, min_dist=min_dist,
    kws_pca=False,  # uses pre-existing batch-effect/integration-corrected PCA
    layer="log1p", use_rapids=use_rapids,
    kws_neighbors=dict(n_neighbors=n_neighbors))
self.cluster(col_celltype=cct, **kws_cl_overall)  # cluster

# Store Clustering Keywords & Plot UMAP by Groups/Samples
self.rna.obs = self.rna.obs.assign(
    **{"kws_cluster_overall": str(kws_cl_overall),
       f"{cct}_log2fc_threshold": t_lfc, f"{cct}_p_threshold": t_p})
factors = [col_batch, col_condition, cct, col_sample, col_subject]
factors = [i for i in factors if i is not None and i in self.rna.obs]
_ = self.plot(kind="umap", wspace=0.5, palette="tab20", color=factors)  # UMAP

# Print Cluster Cell Ns (Overall + by Grouping Variables)
print("\n\n\n")
print(self.rna.obs[cct].value_counts().to_frame("n_cells"))  # N/cluster
for q in [i for i in factors if i != cct]:
    print("\n\n\n\n")
    print(self.rna.obs.groupby(q).apply(lambda x: x[
        cct].value_counts(), include_groups=False).unstack(1))  # by group

# DEGs (One Cluster versus All)
print("\n>>>Finding cluster DEGs...\n\n")
self.find_markers(col_celltype=cct, plot="dot")  # DEGs by cluster
markers_df = self.get_markers_df(
    n_genes=None, col_celltype=cct, p_threshold=t_p,
    log2fc_threshold=t_lfc, log2fc_threshold_abs=False)

# Plot GEX of Predefined Markers
print("\n>>>Plotting marker expression by Leiden cluster...\n\n")
# _ = self.plot(genes=mks_c, figsize=(15, 15), vmax=0.7, layer="scaled",
#               show_gene_labels=False, standard_scale="var", kind="matrix")
fig = scflow.pl.plot_matrix_marsilea(self.rna, mksn, cct, figsize=(60, 5))
ffp = os.path.join(dir_results, f"gex___{'_'.join(batches)}__{cct}.jpg")
if os.path.exists(ffp) is False or overwrite is True:
    fig.figure.savefig(ffp, bbox_inches="tight", pad_inches=0.5)
markers_df

# Annotate

Annotate cell types with various methods

## Annotate by Marker Gene Overlap

File from https://github.com/nasa/GeneLab_Data_Processing/blob/master/scRNAseq/10X_Chromium_3prime_Data/GeneLab_CellType_GeneMarkers/GL-DPPD-7111_GeneMarker_Files/GL-DPPD-7111_Mmus_Brain_CellType_GeneMarkers.csv

**Example of the Expected Marker Definition Format**
```
mksn = {
    "CD4 T cells": {"IL7R"},
    "CD14+ Monocytes": {"CD14", "LYZ"},
    "B cells": {"MS4A1"},
    "CD8 T cells": {"CD8A"},
    "NK cells": {"GNLY", "NKG7"},
    "FCGR3A+ Monocytes": {"FCGR3A", "MS4A7"},
    "Dendritic Cells": {"FCER1A", "CST3"},
    "Megakaryocytes": {"PPBP"},
}
```

In [ ]:
# Annotate
method = "overlap_coef"
# method = "overlap_count"
# method = "jaccard"
marker_matches = self.annotate(
    mksn,
    # celltypes_superhierarchical=celltypes_superhierarchical,
    col_celltype=col_celltype, col_celltype_new="annotation_by_overlap",
    # top_n_markers=25,  # can only have this one or `adj_pval_threshold`
    adj_pval_threshold=1e-5,
    method=method, overwrite=True)

# Perform Any Pre-Specified Renaming of Labels
if rename_marker_based_annotation is not None:
    self.rna.obs.loc[:, "annotation_by_overlap"] = self.rna.obs[
        "annotation_by_overlap"].replace(rename_marker_based_annotation)

# DEGs
self.find_markers(col_celltype="annotation_by_overlap")

# Print & Plot Results
try:
    fig = scflow.pl.plot_matrix_marsilea(
        self.rna, genes=mksn,
        col_celltype="annotation_by_overlap", figsize=(60, 5))
    ffp = os.path.join(dir_results, (f"gex___{'_'.join(batches)}__"
                                     "annotation_by_overlap.jpg"))
    if os.path.exists(ffp) is False or overwrite is True:
        fig.figure.savefig(ffp, bbox_inches="tight", pad_inches=0.5)
except Exception as err:
    print(f"\n\nCould not produce Marsilea version of marker plot: {err}")
self.plot(kind="umap", color="annotation_by_overlap", wspace=0.4)
print("\n\n\n")
print(self.rna.obs.groupby(col_batch).apply(lambda x: round(x[
    "annotation_by_overlap"].value_counts(
        normalize=True) * 100, 2), include_groups=False).unstack(0))
round(self.rna.obs[[col_celltype, "annotation_by_overlap"]
                   ].value_counts(normalize=True)* 100, 2).sort_values()
_ = self.plot(genes=mksn, show_gene_labels=False, layer="log1p",
              col_celltype="annotation_by_overlap", standard_scale="var",
              matrix=dict(figsize=(15, 15)), heat=dict(figsize=(25, 20)))
marker_matches.round(0 if method == "overlap_count" else 2)

## Annotate with CellAssign

By individual cell, so not reliant on clustering

In [ ]:
# Annotate
ffp = os.path.join(
    dir_results, f"gex___{'_'.join(batches)}__annotation_cellassign.jpg")
ffp = ffp if (os.path.exists(ffp) is False or overwrite is True) else True
results_cellassign = self.annotate(
    mksn, overwrite=True, use_cellassign=True, layer="counts",
    col_celltype_new="annotation_cellassign", col_sample=col_sample,
    covariates_categorical=covariates_categorical,
    covariates_continuous=covariates_continuous, plot=ffp)

# DEGs
self.find_markers(col_celltype="annotation_cellassign")

# Print & Plot Results
self.plot(kind="umap", color="annotation_cellassign", wspace=0.4)
print(self.rna.obs.groupby(col_batch).apply(lambda x: round(x[
    "annotation_cellassign"].value_counts(
        normalize=True) * 100, 2), include_groups=False).unstack(0))
_ = self.plot(genes=mksn, show_gene_labels=False, layer="log1p",
              col_celltype="annotation_cellassign", standard_scale="var",
              matrix=dict(figsize=(15, 15)), heat=dict(figsize=(25, 20)))
print("\n\n\n")
round(self.rna.obs[[col_celltype, "annotation_cellassign"]
                   ].value_counts(normalize=True)* 100, 2).sort_values()

## Annotate with ToppGene

In [ ]:
# # Options
# min_genes = 2  # minimum markers that have to overlap between Leiden & atlas
# remove_strings = ["----L1-6", # "---[|]M.*",
#                   "facs-", "-nan-",
#                   # "-i_Gaba_3-.*",
#                   "Brain_organoid-organoid_Kanton_Nature-Organoid-..-",
#                   # "Non-neuronal-Macroglial-((^|)(Oligo|Astro))+-",
#                   # "-Glut_E.*IL7R",
#                   "cells hierarchy compared to all cells using T-S.*",
#                   ".*-organoid_Tanaka_cellReport-.+-",
#                   "...BrainAtlas -.*", "-eN2.*", "...Sample groups.*",
#                   "...Sample Type, Dataset.*",
#                   "-Neuronal",
#                   " // Primary Cells by Cluster",
#                   ".World...Primary Cells by Cluster",
#                   "Brain_organoid-organoid_Velasco_nature-6_",
#                   "Fetal_brain-fetalBrain_Zhong_nature-....-",
#                   "Somatosensory_Cortex_....-Neuronal-",
#                   "Non-neuronal-Non-dividing-",
#                   "...Sample groups..6 Anatomical region groups., with 5.*",
#                   "Brain_organoid-organoid_Paulsen_bioRxiv-",
#                   "-Glut_E_(THEMIS)", "[(]THEMIS[)]",  # "[|].*",
#                   "- method, tissue, subtissue, age, lineage.*"]
# drop_name_patterns = ["striatum", "globus", "Entopeduncular",
#                       "Substantia_nigra-", "Thalamus-"]
# toppgene_rename_by_pattern = dict(
#     Inhibitory=["Inh(_|ib)", "GABA"], Excitatory=["Excit", "Glut"],
#     # Inhibitory=["Inh(_|ib)"], Excitatory=["Excit"],
#     # # Gabaergic=["GABA"], Glutamatergic=["Glut"],
#     Astrocyte=["Astrocyte","Astroglia", "Macroglial-Astro"],
#     Microglial=["Microglia", "Micro"],
#     Endothelial=["Endothelial"],
#     Oligodendrocyte=[r"^(?=.*oligo)(?!.*poly)(?!.*opc).*"],
#     OPC=["Polydendrocyte", "OPC"])
# drop_regions = [
#     "Mid-temporal_gyrus_(MTG)", "primary_auditory_cortex_(A1C)",
#     "Somatosensory_Cortex_(S1)", "Anterior_Cingulate_gyrus_(CgG)",
#     "Primary_Motor_Cortex_(M1)",
#     "Mid-temporal_gyrus_(MTG)|Mid-temporal_gyrus_(MTG)",
#     "primary_auditory_cortex_(A1C)|primary_auditory_cortex_(A1C)",
#     "Somatosensory_Cortex_(S1)|Somatosensory_Cortex_(S1)",
#     "Anterior_Cingulate_gyrus_(CgG)|Anterior_Cingulate_gyrus_(CgG)",
#     "Primary_Motor_Cortex_(M1)|Primary_Motor_Cortex_(M1)",
#     r"Neuronal|World / ",
#     "Primary_Visual_cortex_(V1C)|Primary_Visual_cortex_(V1C)",
#     "mon",
#     "BMP_responsible_cell|6m", "bearing_cell|6m", "bearing_cell|GW16", "11",
#     "Non-neuronal-Non-dividing",
#     "Frontal_cortex|Frontal_cortex",
#     "Primary_Visual_cortex_(V1C)", "Substantia_nigra",
#     "Thalamus", "Hippocampus", "Frontal_cortex"
# ]  # remove if name is just a region or top-level hierarchical/undesired type
# drop_regions = drop_regions + [f"{i}-Non-neuronal" for i in drop_regions]

# # Query ToppGene
# results_toppgene = scflow.pp.annotate_by_toppgene(
#     markers_dict, remove_strings=remove_strings,
#     species=species, min_genes=min_genes, source_patterns=source_patterns)

# # Remove or Alter Certain Name Patterns
# drop_names = results_toppgene.Name.apply(lambda x: not any((
#     i.lower() in x.lower() for i in drop_name_patterns)))
# results_toppgene = results_toppgene[drop_names]
# rn_tg = results_toppgene.Name.apply(lambda x: {x: " | ".join([
#     j for j in toppgene_rename_by_pattern if any((re.search(i.lower(
#         ), x.lower()) is not None for i in toppgene_rename_by_pattern[
#             j]))])}).apply(lambda x: np.nan if x[list(x.keys())[
#                 0]] == "" else x).dropna().reset_index(drop=True).apply(
#                     lambda x: pd.Series(x)).stack().reset_index(
#                         0, drop=True)  # renaming guide
# results_toppgene = results_toppgene.replace({"Name": dict(rn_tg)})
# results_toppgene = results_toppgene[~results_toppgene.Name.isin(drop_regions)]

# # Map Labels (Plurality Vote If Sufficient or Top)
# top_cs = dict(results_toppgene.groupby(
#     "Gene Set").apply(lambda x: x.Name.iloc[:10].value_counts().index.values[
#         0] if x.Name.iloc[:10].value_counts(
#             normalize=True).iloc[0] >= 0.25 else x.Name[0]))
# print("\n".join([f"{k}: {top_cs[k]}" for k in top_cs]), "\n\n")
# if "annotation_toppgene" in self.rna.obs:
#     self.rna.obs = self.rna.obs.drop("annotation_toppgene", axis=1)
# self.rna.obs = self.rna.obs.join(self.rna.obs[col_celltype].replace(
#     top_cs).to_frame("annotation_toppgene")).loc[self.rna.obs.index]

# # Display Results
# if "annotation_by_overlap" in self.rna.obs:
#     print(round(self.rna.obs[["annotation_toppgene", "annotation_by_overlap"]
#                              ].value_counts(normalize=True).sort_index(
#                                  ) * 100, 2), "\n\n")
# print(round(self.rna.obs["annotation_toppgene"].value_counts(
#     normalize=True) * 100, 2))
# results_toppgene.reset_index("ID", drop=True).drop([
#     "QValueBonferroni", "QValueFDRBY", "QValueFDRBH",
#     "TotalGenes", "Genes"], axis=1)

## Annotate with CellTypist

Can do this method by individual cell (`predicted_labels`), so not reliant on clustering if `majority_voting=False`, but to get both `predicted_labels` and `majority_voting`, you need clustering 

In [ ]:
%%time

# To Aggregate More Specific Cell Types
celltypist_rename = dict(
    Gabaergic=["GABA", "Taba"], Glutamatergic=["Glut"],
    Dopaminergic=["Dopa"], Serotonergic=["Sero"],
    Cholinergic=["Chol"],
    Inhibitory=["Inh"], Excitatory=["Exc", "Ex IMN"],
    Astrocyte=["Astro", "Bergman"],
    Microglia=["Microglia"],
    Monocyte=["Monocyte"],
    Lymphoid=["Lymphoid"],
    # Pericyte=["peri"],
    Vascular=["Endothelial", "Endo", "VLMC", "SMC", "ABC", "peri"],
    Oligodendrocyte=[r"^(?=.*oligo)(?!.*poly)(?!.*opc).*"],
    OPC=["OPC", "Polydend"])
celltypist_rename.update({
    "Neuroepithelial": ["CHOR", "Ependymal", "Tanycyte", "Hypendymal"],
    # "Neuroepithelial (Choroid)": ["CHOR"],
    # "Neuroepithelial (Arachnoid Barrier Cells)": ["ABC"],
    "Macrophage (BAM)": ["BAM"],
    # "Vascular (SMC)": ["SMC"],
    # "Vascular (Leptomeningeal)": ["VLMC"],
    # "Excitatory Immature": ["Ex IMN"],
    "Olfactory": ["OEC"]})

# Run CellTypist
self.rna.X = self.rna.layers["counts"].copy()
sc.pp.normalize_total(self.rna, target_sum=10000)
sc.pp.log1p(self.rna) # copy=True: do not update adata.X
predictions = self.annotate(
    model_celltypist, col_celltype=col_celltype,
    layer=None, col_celltype_new="", majority_voting=True,
    min_prop=0.5, use_GPU=True)
if "majority_voting" in self.rna.obs:
    self.rna.obs.loc[:, "majority_voting_short"] = self.rna.obs[
        "majority_voting"].apply(lambda x: " ".join(x.split(
            " ")[1:]) if all((i in [str(i) for i in np.arange(
                0, 10)] for i in x.split(" ")[
                    0])) else x)  # drop pointless #s in front of cell types

# Rename Cell Types
rn_ct, rn_cl = [predictions.predicted_labels.groupby(q).apply(
    lambda x: {x.name: " | ".join([j for j in celltypist_rename if any((
        re.search(i.lower(), x.name.lower()) for i in celltypist_rename[
            j]))])}, include_groups=False).apply(lambda x: {list(x.keys())[
                0]: list(x.keys())[0]} if x[list(x.keys())[
                    0]] == "" else x).apply(lambda x: pd.Series(x)).stack(
                        ).reset_index(0, drop=True) for q in [
                            "majority_voting", "predicted_labels"]]
rn_ct["321 Astroependymal NN"] = "Neuroepithelial"  # override
rn_cl["321 Astroependymal NN"] = "Neuroepithelial"  # override
rn_ct["337 DC NN"] = "Dendritic"  # override
rn_cl["337 DC NN"] = "Dendritic"  # override

# Rename
if "annotation_majority_voting" in self.rna.obs:
    self.rna.obs = self.rna.obs.drop("annotation_majority_voting", axis=1)
if "annotation_predicted_labels" in self.rna.obs:
    self.rna.obs = self.rna.obs.drop("annotation_predicted_labels", axis=1)
self.rna.obs = self.rna.obs.join(self.rna.obs.replace({
    "majority_voting": dict(rn_ct)})["majority_voting"].to_frame(
        "annotation_majority_voting"))
self.rna.obs = self.rna.obs.join(self.rna.obs.replace({
    "predicted_labels": dict(rn_cl)})["predicted_labels"].to_frame(
        "annotation_predicted_labels"))

# Collapse Neuronal Cells
self.rna.obs["annotation_predicted_labels_collapsed"] = self.rna.obs[
    "annotation_predicted_labels"].apply(lambda x: "Neuron" if any((
        i.lower() in x.lower() for i in [
            "gaba", "glut", "cholin", "dopa",
            "serot", "excit", "inhib"])) else x)

# Reset Layer & Display Markers
self.rna.X = self.rna.layers["scaled"].copy()
for x in [f"annotation_{i}" for i in ["majority_voting", "predicted_labels"]]:
    if x in self.rna.obs:
        _ = self.plot(genes=mks_cc, figsize=(15, 10), layer="log1p",
                      title=x, col_celltype=x, standard_scale="var",
                      show_gene_labels=False, kind="matrix")

# Display Cell Type Composition & Renaming Mapping
for x in ["annotation_predicted_labels", "annotation_majority_voting"]:
    if x in self.rna.obs:
        print(round(100 * self.rna.obs[[col_batch, x]].groupby(
            col_batch).value_counts(normalize=True), 2), "\n\n\n")
        ffp = os.path.join(dir_results, f"gex___{'_'.join(batches)}__{x}.jpg")
        if overwrite is True or not os.path.exists(ffp):
            fig.figure.savefig(ffp, bbox_inches="tight", pad_inches=0.5)
ct_mapping = pd.concat([rn_cl, rn_ct])
ct_mapping[~(ct_mapping.index.duplicated(
    keep="last") & ct_mapping.duplicated(keep="last"))].sort_values()

## Annotate with Map My Cells

- Make sure to run the following bash commonds after activating the conda environment used for this notebook.

- Pull [`cell_type_mapper`](https://github.com/AllenInstitute/cell_type_mapper) from GitHub (clone into your home directory): `cd && git clone git@github.com:AllenInstitute/cell_type_mapper.git`

- Navigate to that directory and run `pip install .`

- Navigate to the folder containing this notebook.

- Install ABC Atlas (while in same directory as this notebook): `pip install -U git+https://github.com/alleninstitute/abc_atlas_access >& scratch/junk.txt`

- Pull lookup files (while in same directory as this notebook):
```
cd resources
wget https://allen-brain-cell-atlas.s3-us-west-2.amazonaws.com/mapmycells/WMB-10X/20240831/mouse_markers_230821.json
wget https://allen-brain-cell-atlas.s3-us-west-2.amazonaws.com/mapmycells/WMB-10X/20240831/precomputed_stats_ABC_revision_230821.h5
```

---

Note: To use GPU + Torch, you may need to alter the file "cell_type_mapper/src/cell_type_mapper/cell_by_gene/cell_by_gene.py" line `np.where(np.logical_not(np.isfinite(data)))[0]` to read instead

```
try:
    nan_rows = np.where(
        np.logical_not(np.isfinite(data.cpu().numpy())))[0]
except Exception:
    nan_rows = np.where(np.logical_not(np.isfinite(data)))[0]
```

You may have to run the following code in this notebook:

```
os.environ["NUMEXPR_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
```

and

    `_correlation_dot_gpu()` in distance_utils.py change
    `correlation = torch.matmul(arr0, arr1)` to

```
try:
    correlation = torch.matmul(arr0, arr1)
except RuntimeError as err:
    if "CUBLAS_STATUS_NOT_INITIALIZED" in str(err):
        arr0_cpu = arr0.cpu()
        arr1_cpu = arr1.cpu()
        correlation = torch.matmul(arr0_cpu, arr1_cpu).to(arr0.device)
    else:
        raise
```

to manage processes/memory.

In [ ]:
# %%time

# # Write File to Use as Input for Map My Cells
# if overwrite is True or not os.path.exists(file_new):
#     os.makedirs("data", exist_ok=True)
#     self.rna.X = self.rna.layers["counts"]
#     self.rna.write_h5ad(file_new)
# else:
#     raise ValueError("Must be able to overwrite to run Map My Cells.")

# # Run Map My Cells
# self.rna = scflow.pp.run_mapbraincells(
#     file_new, map_my_cells_source=map_my_cells_source,
#     dir_scratch="scratch", dir_resources="resources",
#     validate_output_file="scratch/tmp.h5ad",  # map_to_ensembl=True,
#     map_my_cells_region_keys=map_my_cells_region_keys,
#     map_my_cells_cell_keys=map_my_cells_cell_keys, verbose_stdout=True,
#     n_processors=4, chunk_size=5000, max_gb=5)

# View Results
# _ = self.plot(kind="umap", color=["cellmap_class_name"])
# if "annotation_toppgene" in self.rna.obs:
#     print(self.rna.obs[["cellmap_class_name", "annotation_toppgene"]
#                        ].value_counts().sort_index())
# self.rna.obs[[i for i in self.rna.obs if "cellmap" in i and "ori" not in i]]

# Annotation QC

Also try:

```
sc.pl.rank_genes_groups_dotplot(
    self.rna, key=f"rank_genes_groups_{col_celltype}", dendrogram=False)
sc.pl.rank_genes_groups_heatmap(
    self.rna, key=f"rank_genes_groups_{col_celltype}", dendrogram=False)
sc.pl.rank_genes_groups_matrixplot(
    self.rna, key=f"rank_genes_groups_{col_celltype}", dendrogram=False)
```

## Compare Annotations

In [ ]:
# cols = [col_celltype,
#         "annotation_predicted_labels", "annotation_majority_voting",
#         "annotation_scanvi", "annotation_by_overlap", "annotation_toppgene"]
cols = ["annotation_by_cellassign", "annotation_scanvi",
        "annotation_predicted_labels_collapsed", "annotation_majority_voting",
        # "annotation_toppgene",
        "annotation_by_overlap"]
# cols += [i for i in [
#     "cellmap_class_name", "cellmap_subclass_name"] if i in self.rna.obs]
cols = [i for i in cols if i in self.rna.obs]

# Plot UMAPs
self.plot(kind="umap", color=cols, wspace=0.6)

# Plot Markers
for x in cols:
    _ = self.plot(genes=mks_cc, figsize=(15, 10), layer="log1p",
                  title=x, col_celltype=x, standard_scale="var",
                  show_gene_labels=False, kind="matrix")

# Compare
# self.rna.obs[cols].value_counts().reset_index().groupby(cols[0]).apply(
#         lambda x: x.sort_values("count", ascending=False).reset_index(
#                 drop=True), include_groups=False).reset_index(
#                         -1, drop=True).set_index(cols[1:], append=True)
self.rna.obs[cols].groupby(cols[0]).apply(
        lambda x: round(100 * x.value_counts(normalize=True), 1).sort_index(
                ).sort_values(ascending=False), include_groups=False)

## Separate Cell Type-Specific Marker Plots

### Scanpy

In [ ]:
c_t = "annotation_scanvi"
for x in mks_c:
    _ = self.plot(genes=mks_c[x], figsize=(15, 10), layer="log1p",
                  title=f"{x} Marker Expression in {c_t} Clusters",
                  col_celltype=c_t, standard_scale="var", kind="matrix")
if "Neuron" in mks_c:
    mks_c_r = ["Hap1", "Myt1l", "Alcam", "Rph3a", "Bex2", "Pcp4", "Rtn1",
               "Agap2", "Gap43", "Rab3c", "S100b", "Ly6h", "Ahi1", "Stmn3"]
    mks_c_r = {"Neuron": [i for i in mks_c["Neuron"] if i not in mks_c_r],
               "Neuron-OPC Overlap": mks_c_r}
    _ = self.plot(genes=mks_c_r, figsize=(15, 10), layer="log1p",
                  title=f"Neuron-OPC Marker Expression in {c_t} Clusters",
                  col_celltype=c_t, standard_scale="var", kind="matrix")

### Marsilea

In [ ]:
for x in cols:
    if x in self.rna.obs:
        print(x)
        fig = scflow.pl.plot_matrix_marsilea(
            self.rna, mksn, x, figsize=(60, 5))

## Percent Non-Zero Marker Expression by Cell Type

Example code to probe specific annotation scheme/cell type:

```
print(gex.loc["annotation_scanvi"]["Macrophage (BAM)"].stack(
    ).sort_values(ascending=False))
print(gex.loc["annotation_scanvi"]["Microglial (DAM)"].stack(
    ).sort_values(ascending=False))
```

In [ ]:
self.plot(genes=mks_c, col_celltype=cols[0], figsize=(40, 5),
          subset=self.rna.obs[cols[0]].isin(myeloid), kind="dot")

gex = pd.concat([pd.concat({c: 100 * (self.get_gex_matrix(mks_collapsed[
    c], layer="counts") > 0).join(self.rna.obs[q]).groupby(q).mean(
        ) for c in mks_collapsed}, axis=1).rename_axis([
            "Marker Group", "Gene"], axis=1) for q in cols],
                keys=cols, names=["Annotation"])
gex

## Main Annotation DEGs

In [ ]:
self.find_markers(col_celltype=cols[0], rankby_abs=True,
                  kws_plot=dict(dendrogram=False), inplace=False)

## OSD-927-932-Specific

### Disease-Associated Microglia

Disease-Associated Microglia Markers (see [here](https://pmc.ncbi.nlm.nih.gov/articles/PMC12539720/)):

* Upregulated: Apoe, Lpl, Cst7, Ctsd, Trem2, Tyrobp
* Downregulated: P2ry12/P2ry13, Cx3cr1, Tmem119

In [ ]:
gdam = {"Up": ["Apoe", "Lpl", "Cst7", "Ctsd", "Trem2", "Tyrobp"],
        "Down": ["P2ry12", "P2ry13", "Cx3cr1", "Tmem119"]}
gdam = {k: [g for g in gdam[k] if g in self.rna.var_names] for k in gdam}
self.plot(genes=gdam, col_celltype="annotation_scanvi", figsize=(8, 5),
          subset=self.rna.obs["annotation_scanvi"].isin(myeloid), kind="dot",
          title="Disease-Associated Microglia Markers", layer="scaled")

### Myeloid DEGs

Use

`self.rna.uns["rank_genes_groups_annotation_scanvi_myeloid"]["pts"]`

to get the percent of cells expressing a given gene for each cluster.

Another example for astrocytes & neurons:

```
k_a = f"rank_genes_groups_annotation_by_overlap_neuron_astrocyte"
self.find_markers(col_celltype="annotation_by_overlap", rankby_abs=True,
                  plot=True, key_added=k_a,
                  reference="Neuron", method="logreg",
                  groups=["Neuron", "Astrocyte | Neuron", "Astrocyte"])
```

In [ ]:
grpsm = [m for m in myeloid if m in self.rna.obs[
    "annotation_scanvi"].to_list()]
self.find_markers(col_celltype="annotation_scanvi", rankby_abs=True,
                  kws_plot=dict(dendrogram=False, figsize=(15, 15)),
                  plot=True, method="logreg",
                  groups=grpsm, reference=myeloid[0],
                  key_added=f"rank_genes_groups_annotation_scanvi_myeloid")

### Marker Specificity

In [ ]:
c_l = "annotation_scanvi"
markers_specific = {
    "Microglial": ["P2ry12"]
}
for c in markers_specific:
    print(f"\n\n{'=' * 80}\n{c}\n{'=' * 80}\n\n")
    for ggg in markers_specific[c]:
        print(f"\n\n{'*' * 40}\n{ggg}\n{'*' * 40}\n\n")
        col_cats = [col_material, col_condition, c_l]
        marker_nonzero = self.rna.obs[col_cats].join(
            (self.get_gex_matrix(ggg, layer="counts") > 0))
        marker_nonzero_p = 100 * marker_nonzero.groupby(col_cats).apply(
            lambda x: x[ggg].mean(), include_groups=False).unstack(1)
        marker_nonzero_c = marker_nonzero.groupby(col_cats).sum()[
            ggg].unstack(1)
        print(marker_nonzero_c)
        print(marker_nonzero_p.sort_values(
            marker_nonzero_c.columns[0], ascending=False).round(2))
        # marker_nonzero_p[marker_nonzero_p > 0].round(2).unstack(
        #     0).unstack(0).replace(np.nan, "")

# Final Write

To reload later and do downstream parts of this notebook, run through the "Setup" section, stopping before the "Load Sample Data" section, then run this code:

```
self = scflow.Rna(file_new, col_sample=col_sample, col_subject=col_subject, 
                  col_batch=col_batch, integrated=True, col_celltype="leiden")
```

This code is also included (commented out) in the "Setup Marker Dictionaries" section (which you should also run after reloading the data), so you can just un-comment out the code in that block, run the whole cell, then re-comment out the part reloading the data (as written above).

In [ ]:
%%time

# Condition Variables
self.rna.obs["Spaceflight"] = (
    self.rna.obs[col_condition] == key_treatment).astype(int)
self.rna.obs["Aged"] = (self.rna.obs[col_age] == self.rna.obs[col_age].max(
    )).astype(int)
self.rna.obs["Age"] = self.rna.obs[col_age].astype(str) + " Months"
self.rna.obs["Condition"] = self.rna.obs["Factor Value[Spaceflight]"].copy()
self.rna.obs["n_cells_original_sample"] = self.rna.obs[
    f"n_cells_original_{col_sample}"].copy()
self.rna.obs["Region"] = self.rna.obs["Brain_Region"].str.capitalize().copy()
print(self.rna.obs[["Spaceflight", col_condition]].value_counts())
print(self.rna.obs[["Aged", col_age]].value_counts())

# Write h5ad
# self.rna.X = self.rna.layers["counts"].copy()
if overwrite is True or not os.path.exists(file_new):
    print("\n\n", f"Writing file to {file_new}...")
    self.rna.write_h5ad(file_new)

## Write Version Compatible with Older Packages
# adata = self.rna.copy()
# adata.uns = {}
# # adata.write_h5ad(os.path.splitext(file_new)[0] + "_compatible.h5ad")

# Send Email with Output When Done
if html_out is not None and (overwrite or not os.path.exists(html_out)):
    os.system(f"jupyter nbconvert --to html {cur_file} --output {html_out}")
    if email is not None:
        os.system(f"echo 'yay' | mutt -s 'JOB DONE' -a {html_out} -- {email}")